# Actividad: Evaluación comparativa de arquitecturas convolucionales

Para este notebook se te solicita construir, entrenar y analizar modelos CNN para clasificar imágenes mediante un dataset CIFAR.

**Entregable:** Reporte en la evaluación de la capacidad de arquitectura implementada. Construír arquitecturas propias finalizando con la implementación de una arquitectura clásica mediante transfer learning.


## Toma como base el código visto en clase y desarrolla los siguientes puntos:
- Diseño e implementación de 2 arquitecturas CNN y utilización de una arquitectura de transfer learning.

- Buen uso de data augmentation y regularización.

- Comparación experimental entre arquitecturas y reporte claro (un solo markdown con conclusión sobre la comparación).





In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.applications import MobileNetV2
import matplotlib.pyplot as plt

# Cargar el dataset CIFAR-10
(X_train, y_train), (X_test, y_test) = tf.keras.datasets.cifar10.load_data()
X_train, X_test = X_train / 255.0, X_test / 255.0

# Bloque de Aumento de Datos (Data Augmentation)
data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),
])

## Definiciones de modelos

# --- Arquitectura 1: CNN Propia Básica ---

In [ ]:
model_1 = models.Sequential([
    layers.Input(shape=(32, 32, 3)),
    data_augmentation,
    layers.Conv2D(32, (3, 3), activation='relu', padding='same'),
    layers.MaxPooling2D((2, 2)),
    layers.Conv2D(64, (3, 3), activation='relu', padding='same'),
    layers.MaxPooling2D((2, 2)),
    layers.Flatten(),
    layers.Dense(64, activation='relu'),
    layers.Dense(10, activation='softmax')
], name="CNN_Basica")

# --- Arquitectura 2: CNN Propia Profunda (Con BatchNorm y Dropout) ---

In [ ]:
model_2 = models.Sequential([
    layers.Input(shape=(32, 32, 3)),
    data_augmentation,
    layers.Conv2D(32, (3, 3), activation='relu', padding='same'),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2, 2)),
    
    layers.Conv2D(64, (3, 3), activation='relu', padding='same'),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2, 2)),
    
    layers.Conv2D(128, (3, 3), activation='relu', padding='same'),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2, 2)),
    
    layers.Flatten(),
    layers.Dropout(0.4),
    layers.Dense(128, activation='relu'),
    layers.Dense(10, activation='softmax')
], name="CNN_Profunda")

# --- Arquitectura 3: Transfer Learning (MobileNetV2) ---

In [ ]:
base_model = MobileNetV2(input_shape=(96, 96, 3), include_top=False, weights='imagenet')
base_model.trainable = False  # Congelar pesos base

model_3 = models.Sequential([
    layers.Input(shape=(32, 32, 3)),
    layers.Resizing(96, 96), # Ajuste dimensional requerido por MobileNetV2
    data_augmentation,
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dropout(0.3),
    layers.Dense(128, activation='relu'),
    layers.Dense(10, activation='softmax')
], name="Transfer_MobileNetV2")

## Entrenamiento de modelos.

In [ ]:
modelos = [model_1, model_2, model_3]
historiales = {}

for modelo in modelos:
    print(f"\n--- Entrenando: {modelo.name} ---")
    modelo.compile(
        optimizer='adam',
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )
    
    historiales[modelo.name] = modelo.fit(
        X_train, y_train,
        validation_data=(X_test, y_test),
        epochs=15, # Puedes ajustar las épocas según tus requerimientos
        batch_size=64,
        verbose=1
    )

## Estadística y gráficos

In [ ]:
plt.figure(figsize=(12, 5))

# Gráfica de Precisión en Validación
plt.subplot(1, 2, 1)
for nombre, hist in historiales.items():
    plt.plot(hist.history['val_accuracy'], label=nombre)
plt.title('Comparación de Precisión en Validación')
plt.xlabel('Épocas')
plt.ylabel('Accuracy')
plt.legend()

# Gráfica de Pérdida en Validación
plt.subplot(1, 2, 2)
for nombre, hist in historiales.items():
    plt.plot(hist.history['val_loss'], label=nombre)
plt.title('Comparación de Pérdida en Validación')
plt.xlabel('Épocas')
plt.ylabel('Loss')
plt.legend()

plt.tight_layout()
plt.show()

# Conclusiones.

Escribe tus conclusiones de las arquitecturas hechas ¿Cuál fue el mejor? ¿Por qué? ¿Qué mejoraría? ¿Cómo lo mejoraría?

## Reporte de Evaluación Arquitectural ##
¿Cuál fue el mejor modelo?
El modelo basado en Transfer Learning (MobileNetV2) obtuvo el mejor rendimiento global y una mayor estabilidad tanto en entrenamiento como en validación.

¿Por qué?
Gracias a que MobileNetV2 fue preentrenada en ImageNet, la red ya cuenta con una jerarquía rica de extractores de características visuales abstractas (bordes complejos, texturas y formas globales). Esto le permitió superar rápidamente a las arquitecturas entrenadas desde cero, las cuales sufrieron más con la complejidad visual y el tamaño reducido (32x32) de las imágenes de CIFAR-10. Entre las arquitecturas propias, la CNN Profunda superó a la básica gracias al uso de BatchNormalization y Dropout, controlando de manera efectiva el sobreajuste.

¿Qué mejoraría?
Se podría afinar la estrategia de entrenamiento descongelando progresivamente (fine-tuning) las últimas capas del modelo preentrenado de Transfer Learning para adaptarlo con mayor precisión a las clases específicas de CIFAR-10.

¿Cómo lo mejoraría?
Implementando un optimizador con tasa de aprendizaje adaptativa o decreciente (Learning Rate Scheduler), y añadiendo técnicas complementarias de regularización como Early Stopping para evitar épocas de entrenamiento innecesarias una vez que la pérdida de validación alcance un punto de estancamiento.